# Aula 13 - One-Class SVM: Ilustrações Didáticas

Este notebook gera figuras Plotly para explicar o One-Class SVM passo a passo:
1. Dados não lineares (círculos concêntricos)
2. Transformação via kernel polinomial
3. O truque do kernel (kernel trick)
4. Seleção de support vectors via otimização
5. Classificação de novos pontos

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.utils import PlotlyJSONEncoder
from pathlib import Path
import json
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DRACULA = {
    'bg': '#282a36',
    'fg': '#f8f8f2',
    'cyan': '#8be9fd',
    'green': '#50fa7b',
    'pink': '#ff79c6',
    'red': '#ff5555',
    'yellow': '#f1fa8c',
    'purple': '#bd93f9',
    'muted': '#6272a4',
}

def apply_dracula(fig, title=None):
    fig.update_layout(
        title=title,
        paper_bgcolor=DRACULA['bg'],
        plot_bgcolor=DRACULA['bg'],
        font=dict(color=DRACULA['fg']),
        margin=dict(l=60, r=30, t=60, b=55),
        showlegend=True,
        legend=dict(orientation='h', y=-0.25),
    )
    fig.update_xaxes(gridcolor='#44475a', zerolinecolor=DRACULA['muted'])
    fig.update_yaxes(gridcolor='#44475a', zerolinecolor=DRACULA['muted'])
    return fig

def clean_none(obj):
    if isinstance(obj, dict):
        return {k: clean_none(v) for k, v in obj.items() if v is not None}
    if isinstance(obj, list):
        return [clean_none(v) for v in obj]
    return obj

def slide_export_path():
    cwd = Path.cwd().resolve()
    if cwd.name == 'notebooks' and cwd.parent.name == 'ciencia-dados':
        return cwd.parent / 'images' / 'plotly' / 'aula13_oneclass_svm_figures.js'
    return cwd / 'ciencia-dados' / 'images' / 'plotly' / 'aula13_oneclass_svm_figures.js'

def write_plotly_js(figures, output_path):
    lines = []
    for name, fig in figures.items():
        payload = clean_none(fig.to_dict())
        traces = json.dumps(payload['data'], ensure_ascii=False, cls=PlotlyJSONEncoder)
        layout = json.dumps(payload['layout'], ensure_ascii=False, cls=PlotlyJSONEncoder)
        lines.append(f"const {name}_TRACES = {traces};\n")
        lines.append(f"const {name}_LAYOUT = {layout};\n")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text('\n'.join(lines), encoding='utf-8')


## 1. Dados sintéticos: círculos concêntricos

Pontos internos (raio pequeno) = normais.
Pontos externos (raio grande) = anomalias.
Não são linearmente separáveis no espaço original.

In [ ]:
# Gerar círculos concêntricos
n_normal = 120
n_anom = 30

# Normais: raio entre 1 e 2.5
theta_n = np.random.uniform(0, 2*np.pi, n_normal)
r_n = np.random.uniform(1.0, 2.5, n_normal) + np.random.normal(0, 0.15, n_normal)
X_normal = np.column_stack([r_n * np.cos(theta_n), r_n * np.sin(theta_n)])

# Anomalias: raio entre 4 e 5.5
theta_a = np.random.uniform(0, 2*np.pi, n_anom)
r_a = np.random.uniform(4.0, 5.5, n_anom) + np.random.normal(0, 0.2, n_anom)
X_anom = np.column_stack([r_a * np.cos(theta_a), r_a * np.sin(theta_a)])

X = np.vstack([X_normal, X_anom])
y = np.array([0]*n_normal + [1]*n_anom)  # 0=normal, 1=anomalia

print(f"Total: {len(X)} pontos | Normais: {n_normal} | Anomalias: {n_anom}")

## 2. Figura: Dados originais — não linearmente separáveis

In [ ]:
fig_original = go.Figure()

fig_original.add_trace(go.Scatter(
    x=X_normal[:, 0], y=X_normal[:, 1],
    mode='markers', name='Normal (raio pequeno)',
    marker=dict(color=DRACULA['cyan'], size=10, opacity=0.7),
))
fig_original.add_trace(go.Scatter(
    x=X_anom[:, 0], y=X_anom[:, 1],
    mode='markers', name='Anomalia (raio grande)',
    marker=dict(color=DRACULA['red'], size=12, symbol='x', line=dict(width=3)),
))

# Linha de tentativa linear (falha)
fig_original.add_shape(type='line', x0=-6, y0=0, x1=6, y1=0,
                       line=dict(color=DRACULA['yellow'], width=2, dash='dash'))
fig_original.add_annotation(x=4, y=0.5, text='Reta linear não separa',
                           font=dict(color=DRACULA['yellow'], size=14), showarrow=False)

fig_original.update_xaxes(title='x₁', range=[-6.5, 6.5])
fig_original.update_yaxes(title='x₂', range=[-6.5, 6.5])
fig_original.update_layout(width=700, height=600)
fig_original = apply_dracula(fig_original, "Dados originais: círculos concêntricos não são linearmente separáveis")
fig_original.show()

## 3. Figura: Transformação polinomial — agora é separável

Transformação: tomamos a distância ao centro ao quadrado r² = x₁² + x₂².
No novo espaço (x₁, r²), os círculos viram faixas horizontais — separáveis por uma reta.

In [ ]:
r2 = X[:, 0]**2 + X[:, 1]**2

fig_transform = go.Figure()

fig_transform.add_trace(go.Scatter(
    x=X_normal[:, 0], y=r2[:n_normal],
    mode='markers', name='Normal',
    marker=dict(color=DRACULA['cyan'], size=10, opacity=0.7),
))
fig_transform.add_trace(go.Scatter(
    x=X_anom[:, 0], y=r2[n_normal:],
    mode='markers', name='Anomalia',
    marker=dict(color=DRACULA['red'], size=12, symbol='x', line=dict(width=3)),
))

# Reto separadora no espaço transformado
fig_transform.add_hline(y=10, line_color=DRACULA['green'], line_width=3, line_dash='solid')
fig_transform.add_annotation(x=4, y=11, text='Agora separável por uma reta!',
                           font=dict(color=DRACULA['green'], size=15), showarrow=False)

fig_transform.update_xaxes(title='x₁ (original)')
fig_transform.update_yaxes(title='r² = x₁² + x₂² (transformado)', range=[0, 35])
fig_transform.update_layout(width=700, height=600)
fig_transform = apply_dracula(fig_transform, "Espaço transformado: distância ao centro torna os grupos separáveis")
fig_transform.show()

## 4. Figura: O truque do kernel (kernel trick)

Não precisamos calcular φ(x) explicitamente.
Basta calcular K(a, b) = (a · b)², que é igual a φ(a) · φ(b), mas muito mais rápido.
Para dimensão d, φ(x) teria ~d²/2 dimensões; o kernel sempre precisa apenas do produto interno original.

In [ ]:
fig_kernel = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Cálculo direto (φ explícito)', 'Via kernel K(a,b) = (a·b)²'),
    horizontal_spacing=0.12
)

# Escolher dois pontos de exemplo
a = X_normal[0]
b = X_normal[1]

# φ(a) = [a1², sqrt(2)*a1*a2, a2²]
phi = lambda v: np.array([v[0]**2, np.sqrt(2)*v[0]*v[1], v[1]**2])
phi_a = phi(a)
phi_b = phi(b)
dot_direct = float(np.dot(phi_a, phi_b))
dot_kernel = float(np.dot(a, b)**2)

# Subplot 1: barras mostrando φ(a) e φ(b) + produto interno
categories = ['φ₁=a₁²', 'φ₂=√2·a₁a₂', 'φ₃=a₂²']
fig_kernel.add_trace(go.Bar(
    x=categories, y=phi_a, name='φ(a)', marker_color=DRACULA['cyan'], opacity=0.7
), row=1, col=1)
fig_kernel.add_trace(go.Bar(
    x=categories, y=phi_b, name='φ(b)', marker_color=DRACULA['pink'], opacity=0.7
), row=1, col=1)
fig_kernel.add_annotation(
    x=1, y=max(max(phi_a), max(phi_b))*1.1,
    text=f'φ(a)·φ(b) = {dot_direct:.2f}',
    showarrow=False, font=dict(color=DRACULA['fg'], size=14), row=1, col=1
)

# Subplot 2: barras mostrando a, b e (a·b)²
cats2 = ['a₁', 'a₂', 'b₁', 'b₂']
vals2 = [a[0], a[1], b[0], b[1]]
colors2 = [DRACULA['cyan'], DRACULA['cyan'], DRACULA['pink'], DRACULA['pink']]
fig_kernel.add_trace(go.Bar(
    x=cats2, y=vals2, showlegend=False,
    marker_color=colors2, opacity=0.7
), row=1, col=2)
fig_kernel.add_annotation(
    x=1.5, y=max(vals2)*2.5,
    text=f'(a·b)² = {dot_kernel:.2f}',
    showarrow=False, font=dict(color=DRACULA['fg'], size=14), row=1, col=2
)
fig_kernel.add_annotation(
    x=1.5, y=max(vals2)*1.8,
    text='Mesmo resultado, menos operações!',
    showarrow=False, font=dict(color=DRACULA['green'], size=13), row=1, col=2
)

fig_kernel.update_layout(
    title_text='Kernel trick: evita calcular φ(x) explicitamente',
    paper_bgcolor=DRACULA['bg'], plot_bgcolor=DRACULA['bg'],
    font=dict(color=DRACULA['fg']),
    showlegend=True, legend=dict(orientation='h', y=-0.15),
    margin=dict(l=60, r=30, t=80, b=60),
)
fig_kernel.update_xaxes(gridcolor='#44475a')
fig_kernel.update_yaxes(gridcolor='#44475a')
fig_kernel.update_layout(width=900, height=450)
fig_kernel.show()

## 5. Figura: Support Vectors — os pontos que definem a fronteira

Treinamos um One-Class SVM com kernel polinomial (grau 2).
Os pontos com peso α > 0 são os support vectors (SVs).
Pontos com α = 0 não participam da decisão.

In [ ]:
# Treinar OC-SVM com kernel polinomial nos dados normais apenas
scaler = StandardScaler()
X_train = scaler.fit_transform(X_normal)
X_all_scaled = scaler.transform(X)

modelo = OneClassSVM(kernel='poly', degree=2, gamma=1, coef0=0, nu=0.15)
modelo.fit(X_train)

# Índices dos support vectors dentro de X_train
sv_indices_train = modelo.support_
# Obter os índices correspondentes em X (normais estão nas primeiras n_normal posições)
sv_indices_global = sv_indices_train  # porque X_train vem de X_normal que é prefixo de X

# Pesos (dual_coef_) no OC-SVM shape é (1, n_sv)
alphas = np.abs(modelo.dual_coef_.ravel())
alphas_norm = alphas / alphas.max()  # normalizar para visualização

fig_svs = go.Figure()

# Pontos normais não-SV
normal_not_sv = np.setdiff1d(np.arange(n_normal), sv_indices_global)
fig_svs.add_trace(go.Scatter(
    x=X_normal[normal_not_sv, 0], y=X_normal[normal_not_sv, 1],
    mode='markers', name='Normal (α = 0)',
    marker=dict(color=DRACULA['cyan'], size=8, opacity=0.4),
))

# Anomalias
fig_svs.add_trace(go.Scatter(
    x=X_anom[:, 0], y=X_anom[:, 1],
    mode='markers', name='Anomalia',
    marker=dict(color=DRACULA['red'], size=10, symbol='x', line=dict(width=2)),
))

# Support vectors com tamanho proporcional a α
fig_svs.add_trace(go.Scatter(
    x=X_normal[sv_indices_global, 0], y=X_normal[sv_indices_global, 1],
    mode='markers', name='Support Vector (α > 0)',
    marker=dict(
        color=DRACULA['yellow'],
        size=10 + 25*alphas_norm,
        opacity=0.9,
        line=dict(color=DRACULA['fg'], width=1)
    ),
))

fig_svs.update_xaxes(title='x₁', range=[-6.5, 6.5])
fig_svs.update_yaxes(title='x₂', range=[-6.5, 6.5])
fig_svs.update_layout(width=700, height=600)
fig_svs = apply_dracula(fig_svs, "Support Vectors: apenas os pontos de fronteira definem o modelo")
fig_svs.show()
print(f"Support vectors: {len(sv_indices_global)} de {n_normal} normais")

## 6. Figura: Fronteira de decisão — normal vs anomalia

In [ ]:
# Grid para contorno
xx, yy = np.meshgrid(
    np.linspace(-6, 6, 300),
    np.linspace(-6, 6, 300)
)
grid = np.c_[xx.ravel(), yy.ravel()]
Z = modelo.decision_function(scaler.transform(grid)).reshape(xx.shape)

# Cores para contorno: verde onde f(x) >= 0 (normal), vermelho onde < 0 (anomalia)
fig_front = go.Figure()

fig_front.add_trace(go.Contour(
    x=xx[0], y=yy[:, 0], z=Z,
    contours=dict(start=-0.5, end=0.5, size=0.5, coloring='heatmap'),
    colorscale=[[0, DRACULA['red']], [0.5, DRACULA['yellow']], [1, DRACULA['green']]],
    showscale=False, opacity=0.5,
    name='Região de decisão'
))

# Fronteira f(x) = 0
fig_front.add_trace(go.Contour(
    x=xx[0], y=yy[:, 0], z=Z,
    contours=dict(start=0, end=0, size=0, coloring='lines'),
    line=dict(color=DRACULA['fg'], width=3),
    showscale=False,
    name='Fronteira f(x)=0'
))

# Pontos normais
fig_front.add_trace(go.Scatter(
    x=X_normal[:, 0], y=X_normal[:, 1],
    mode='markers', name='Normal',
    marker=dict(color=DRACULA['cyan'], size=8, opacity=0.7),
))

# Anomalias
fig_front.add_trace(go.Scatter(
    x=X_anom[:, 0], y=X_anom[:, 1],
    mode='markers', name='Anomalia',
    marker=dict(color=DRACULA['red'], size=10, symbol='x', line=dict(width=2)),
))

# Novo ponto de exemplo
x_novo = np.array([[3.0, 2.0], [-1.0, 1.0]])  # um fora, um dentro
pred_novo = modelo.decision_function(scaler.transform(x_novo))
labels_novo = ['Normal (f≥0)', 'Anomalia (f<0)']
colors_novo = [DRACULA['green'], DRACULA['red']]
for i, (pt, lbl, c) in enumerate(zip(x_novo, labels_novo, colors_novo)):
    fig_front.add_trace(go.Scatter(
        x=[pt[0]], y=[pt[1]], mode='markers+text',
        name=lbl, text=[lbl], textposition='top center',
        marker=dict(color=c, size=18, symbol='diamond', line=dict(color=DRACULA['fg'], width=2)),
        textfont=dict(color=c, size=12)
    ))

fig_front.update_xaxes(title='x₁', range=[-6.5, 6.5])
fig_front.update_yaxes(title='x₂', range=[-6.5, 6.5])
fig_front.update_layout(width=700, height=600)
fig_front = apply_dracula(fig_front, "Fronteira de decisão: f(x) = Σ αᵢ K(xᵢ, x) − ρ")
fig_front.show()

## 7. Exportar figuras para JS

In [ ]:
figures = {
    'AULA13_SVM_ORIGINAL': fig_original,
    'AULA13_SVM_TRANSFORMADO': fig_transform,
    'AULA13_SVM_KERNEL_TRICK': fig_kernel,
    'AULA13_SVM_SUPPORT_VECTORS': fig_svs,
    'AULA13_SVM_FRONTEIRA': fig_front,
}

write_plotly_js(figures, slide_export_path())
print(f"Exported {len(figures)} figures to {slide_export_path()}")